In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── 0. Setup & cấu hình ─────────────────────────────────────────────────────
import os, sys, shutil, glob, struct, json, subprocess
import torch

IN_COLAB = "google.colab" in sys.modules      # tự nhận Colab / RunPod

assert torch.cuda.is_available(), "Không thấy GPU (Colab: Runtime > Change runtime type > GPU)"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)

!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg colmap xvfb libgl1 libglib2.0-0
!pip install -q ninja
if not IN_COLAB:
    !pip install -q opencv-python-headless

import numpy as np, cv2
print("nvcc:", shutil.which("nvcc") or "KHÔNG THẤY -> cần template PyTorch '-devel' để compile gsplat")

# ── CẤU HÌNH (đổi SCENE_NAME cho mỗi video) ─────────────────────────────────
SCENE_NAME     = "scene1"
FRAME_FPS      = 5          # số frame / giây khi trích từ video
FRAME_HEIGHT   = 1080        # chiều cao frame (px)
COLMAP_MATCHER = "exhaustive"   # "exhaustive" (ổn định hơn) hoặc "sequential" (nhanh hơn)

BASE          = "/content" if IN_COLAB else "/workspace"   # Colab: /content (mất khi hết phiên)
WORK_DIR      = f"{BASE}/scene3d_work/{SCENE_NAME}"
FRAMES_DIR    = f"{WORK_DIR}/frames"
COLMAP_DIR    = f"{WORK_DIR}/colmap"
GSPLAT_REPO   = f"{BASE}/gsplat"
GSPLAT_OUTPUT = f"{WORK_DIR}/gsplat_output"
OUT_DIR       = f"{BASE}/outputs/{SCENE_NAME}"
VIEWER_DIR    = f"{BASE}/viewer/{SCENE_NAME}"
for d in (WORK_DIR, OUT_DIR, VIEWER_DIR): os.makedirs(d, exist_ok=True)

# Tìm video đã upload (tên gì cũng được) -> input.mp4
VIDEO = f"{WORK_DIR}/input.mp4"
if not os.path.exists(VIDEO):
    found = sorted(sum([glob.glob(f"{WORK_DIR}/*.{e}") for e in ("mp4","mov","mkv","avi","webm","MP4","MOV")], []))
    if found:
        os.symlink(found[0], VIDEO)
    elif IN_COLAB:
        from google.colab import files
        print("Chọn file video để upload...")
        up = files.upload()
        if up: shutil.move(next(iter(up)), VIDEO)
print("Video:", VIDEO, "->", "OK" if os.path.exists(VIDEO) else f"CHƯA CÓ — upload vào {WORK_DIR}/ rồi chạy lại cell này")

GPU: NVIDIA A100-SXM4-40GB | PyTorch 2.11.0+cu130 | CUDA 13.0
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2

Saving video2.mp4 to video2.mp4
Video: /content/scene3d_work/scene1/input.mp4 -> OK


---
## 1. Frame Extraction
ffmpeg trích frame, bỏ 5% frame mờ nhất (phương sai Laplacian thấp), đánh số lại.

In [ ]:
assert os.path.exists(VIDEO), f"Chưa có video trong {WORK_DIR}/"

shutil.rmtree(FRAMES_DIR, ignore_errors=True); os.makedirs(FRAMES_DIR)
r = subprocess.run(["ffmpeg", "-i", VIDEO, "-vf", f"fps={FRAME_FPS},scale=-1:{FRAME_HEIGHT}",
                    "-q:v", "2", "-y", f"{FRAMES_DIR}/%06d.jpg"], capture_output=True, text=True)
if r.returncode != 0: raise RuntimeError(r.stderr[-500:])
frames = sorted(glob.glob(f"{FRAMES_DIR}/*.jpg"))
assert frames, "ffmpeg không tạo ra frame nào"

# Lọc frame mờ
blur = [cv2.Laplacian(cv2.imread(f, cv2.IMREAD_GRAYSCALE), cv2.CV_64F).var() for f in frames]
thr  = np.percentile(blur, 10)
keep = [f for f, b in zip(frames, blur) if b >= thr]
for f, b in zip(frames, blur):
    if b < thr: os.remove(f)

# Đánh số lại, lưu PNG
for i, f in enumerate(keep, 1):
    cv2.imwrite(f"{FRAMES_DIR}/{i:06d}.png", cv2.imread(f)); os.remove(f)

n_frames = len(glob.glob(f"{FRAMES_DIR}/*.png"))
print(f"✓ {n_frames} frames (đã bỏ {len(frames)-len(keep)} frame mờ) -> {FRAMES_DIR}")

✓ 142 frames (đã bỏ 16 frame mờ) -> /content/scene3d_work/scene1/frames


---
## 2. COLMAP (Structure-from-Motion)
Ước lượng vị trí camera và đám mây điểm thưa. Đây là bước quyết định chất lượng: nếu số frame đăng ký được quá thấp thì cần quay lại video.

In [ ]:
os.environ["QT_QPA_PLATFORM"] = "offscreen"

# Chuẩn bị thư mục sạch
colmap_imgs = f"{COLMAP_DIR}/images"
shutil.rmtree(COLMAP_DIR, ignore_errors=True)
shutil.copytree(FRAMES_DIR, colmap_imgs)
if os.path.exists(f"{WORK_DIR}/colmap.db"): os.remove(f"{WORK_DIR}/colmap.db")
sparse_out = f"{COLMAP_DIR}/sparse"; os.makedirs(sparse_out)

print("[1/3] Trích đặc trưng SIFT...")
!colmap feature_extractor \
    --database_path {WORK_DIR}/colmap.db \
    --image_path {colmap_imgs} \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0

print(f"\n[2/3] Matching ({COLMAP_MATCHER})...")
if COLMAP_MATCHER == "exhaustive":
    !colmap exhaustive_matcher --database_path {WORK_DIR}/colmap.db --SiftMatching.use_gpu 0
else:
    !colmap sequential_matcher --database_path {WORK_DIR}/colmap.db --SiftMatching.use_gpu 0 \
        --SequentialMatching.overlap 10 --SequentialMatching.loop_detection 1

print("\n[3/3] Sparse reconstruction...")
!colmap mapper --database_path {WORK_DIR}/colmap.db --image_path {colmap_imgs} --output_path {sparse_out}

# Chọn model có nhiều ảnh đăng ký nhất -> sparse/0
def count_images(d):
    p = f"{d}/images.bin"
    return struct.unpack("<Q", open(p, "rb").read(8))[0] if os.path.exists(p) else 0

models = sorted(m for m in os.listdir(sparse_out) if os.path.isdir(f"{sparse_out}/{m}"))
if not models: raise RuntimeError("COLMAP không dựng được cảnh. Quay chậm hơn, đủ sáng, nhiều chi tiết.")
best = max(models, key=lambda m: count_images(f"{sparse_out}/{m}"))
if best != "0":
    shutil.rmtree(f"{sparse_out}/0", ignore_errors=True)
    shutil.copytree(f"{sparse_out}/{best}", f"{sparse_out}/0")

n_reg = count_images(f"{sparse_out}/0")
n_pts = struct.unpack("<Q", open(f"{sparse_out}/0/points3D.bin", "rb").read(8))[0]
print(f"\n✓ COLMAP: đăng ký {n_reg}/{n_frames} frame ({100*n_reg/n_frames:.0f}%), {n_pts:,} điểm thưa, {len(models)} model")
if n_reg < 0.5 * n_frames:
    print("⚠ Tỉ lệ đăng ký thấp -> cảnh sẽ thủng/nhiễu. Nên quay lại video (chậm, tịnh tiến, nhiều texture).")

Streaming output truncated to the last 5000 lines.
 Initial cost : 0.441355 [px]
   Final cost : 0.403461 [px]
  Termination : Convergence

I20261003 13:02:18.341009  6014 incremental_mapper.cc:40] => Continued observations: 1356
I20261003 13:02:18.347301  6014 incremental_mapper.cc:43] => Added observations: 419
I20261003 13:02:18.723073  6014 misc.cc:205] 
Bundle adjustment report
------------------------
I20261003 13:02:18.723147  6014 bundle_adjustment.cc:942] 
    Residuals : 23302
   Parameters : 2639
   Iterations : 19
         Time : 0.358972 [s]
 Initial cost : 0.375725 [px]
   Final cost : 0.357462 [px]
  Termination : Convergence

I20261003 13:02:18.761956  6014 incremental_mapper.cc:78] => Merged observations: 143
I20261003 13:02:18.762012  6014 incremental_mapper.cc:79] => Completed observations: 114
I20261003 13:02:18.762017  6014 incremental_mapper.cc:81] => Filtered observations: 103
I20261003 13:02:18.762022  6014 incremental_mapper.cc:90] => Changed observations: 0.02

---
## 3. Gaussian Splatting (gsplat)
Cài gsplat, compile CUDA (lần đầu mất vài phút), rồi train trên các frame đã đăng ký. gsplat tự giữ lại 1/8 frame để đánh giá (PSNR/SSIM/LPIPS).

In [ ]:
# Compile CUDA đúng kiến trúc GPU (A40 -> 8.6) để nhanh hơn
cap = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
os.environ["MAX_JOBS"] = str(max(1, min(4, (os.cpu_count() or 2) // 2)))
print("CUDA arch:", os.environ["TORCH_CUDA_ARCH_LIST"], "| MAX_JOBS:", os.environ["MAX_JOBS"])

!pip install -q gsplat==1.3.0
!pip install -q tyro viser "imageio[ffmpeg]" tensorboard "torchmetrics[image]" tqdm scipy nerfview splines pycolmap PyYAML piexif

if not os.path.exists(GSPLAT_REPO):
    !git clone -b v1.3.0 https://github.com/nerfstudio-project/gsplat.git {GSPLAT_REPO}
    !touch {GSPLAT_REPO}/examples/datasets/__init__.py
    !wget -q -O {GSPLAT_REPO}/examples/datasets/colmap.py \
        "https://github.com/nerfstudio-project/gsplat/raw/9b9f98a5b440531376b4a5386aea49f8e820203b/examples/datasets/colmap.py"
    !wget -q -O {GSPLAT_REPO}/examples/exif.py "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"
    !wget -q -O {GSPLAT_REPO}/examples/datasets/exif.py "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"
    !sed -i 's/align_principal_axes/align_principle_axes/g' {GSPLAT_REPO}/examples/datasets/colmap.py
!cd {GSPLAT_REPO} && git submodule update --init --recursive

print("Compile CUDA extension (~5 phút lần đầu)...")
!rm -rf ~/.cache/torch_extensions/
!python -c "import gsplat.cuda._backend; print('✓ CUDA compiled')"

CUDA arch: 8.0 | MAX_JOBS: 4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.5 MB/s eta 0:00:00
Cloning into '/content/gsplat'...
remote: Enumerating objects: 17549, done.
remote: Counting objects: 100% (1196/1196), done.
remote: Compressing objects: 100% (330/330), done.
remote: Total 17549 (delta 928), reused 866 (delta 866), pack-reused 16353 (from

In [ ]:
import shutil, subprocess, sys, os, glob

DATA_DIR = COLMAP_DIR

# ── Kiểm tra dữ liệu COLMAP ──
print("DATA_DIR:", DATA_DIR, "->", os.listdir(DATA_DIR))
assert os.path.isdir(f"{DATA_DIR}/images"), "Thiếu images/"
assert os.path.isdir(f"{DATA_DIR}/sparse/0"), "Thiếu sparse/0 - chạy lại cell COLMAP"
print("Số ảnh:", len(os.listdir(f"{DATA_DIR}/images")))

# ── Tìm trainer ──
cands = glob.glob("/content/**/examples/simple_trainer.py", recursive=True)
assert cands, "Không tìm thấy simple_trainer.py - kiểm tra đã clone gsplat chưa"
TRAINER = cands[0]
print("Trainer:", TRAINER)

# ── Xóa kết quả cũ ──
shutil.rmtree(GSPLAT_OUTPUT, ignore_errors=True)
os.makedirs(GSPLAT_OUTPUT, exist_ok=True)

MAX_STEPS = 40000
cmd = [sys.executable, "-u", TRAINER, "mcmc",          # <-- mcmc, KHÔNG phải default
       "--data_dir", DATA_DIR,
       "--result_dir", GSPLAT_OUTPUT,
       "--data_factor", "1",
       "--max_steps", str(MAX_STEPS),
       "--strategy.cap-max", "3500000",
       "--strategy.refine-stop-iter", "35000",
       "--opacity_reg", "0.01",
       "--scale_reg", "0.01",
       "--antialiased",
       "--eval_steps", str(MAX_STEPS),
       "--save_steps", str(MAX_STEPS),
       "--disable_viewer"]
print(" ".join(cmd), flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1")
proc = subprocess.Popen(cmd, cwd=os.path.dirname(os.path.dirname(TRAINER)),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line.rstrip()[-200:], flush=True)
rc = proc.wait()
assert rc == 0, "Training LỖI - xem log phía trên, đừng chạy cell xuất file"
print("✓ Training xong")

Streaming output truncated to the last 5000 lines.
loss=0.009| sh degree=3| :  92%|█████████▏| 36725/40000 [39:55<04:04, 13.38it/s]
loss=0.010| sh degree=3| :  92%|█████████▏| 36725/40000 [39:55<04:04, 13.38it/s]
loss=0.010| sh degree=3| :  92%|█████████▏| 36727/40000 [39:55<04:02, 13.51it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36727/40000 [39:55<04:02, 13.51it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36727/40000 [39:55<04:02, 13.51it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36729/40000 [39:55<04:05, 13.31it/s]
loss=0.007| sh degree=3| :  92%|█████████▏| 36729/40000 [39:55<04:05, 13.31it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36729/40000 [39:55<04:05, 13.31it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36731/40000 [39:55<04:07, 13.22it/s]
loss=0.008| sh degree=3| :  92%|█████████▏| 36731/40000 [39:55<04:07, 13.22it/s]
loss=0.009| sh degree=3| :  92%|█████████▏| 36731/40000 [39:55<04:07, 13.22it/s]
loss=0.009| sh degree=3| :  92%|█████████▏| 36733/40000 [3

---
## 4. 3D Scene — xuất file
Lấy checkpoint cuối, lọc floaters (Gaussian mờ / quá to / nằm quá xa), rồi xuất `scene.ply` (chuẩn 3DGS) và `scene.splat` (nhẹ, dùng cho web).

In [ ]:
import os, sys, re, glob, json, shutil, subprocess, time
import numpy as np, torch

SH_C0 = 0.28209479177387814
os.makedirs(VIEWER_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ───────── Lọc nhẹ: chỉ bỏ rác thật sự ─────────
def prune_mask(means, log_scales, logit_op):
    keep = (np.isfinite(means).all(1) & np.isfinite(log_scales).all(1) & np.isfinite(logit_op))
    if not keep.any():
        return keep
    op = 1.0 / (1.0 + np.exp(-np.clip(logit_op, -30, 30)))
    k = keep & (op >= 0.005)
    if k.any(): keep = k
    smax = log_scales.max(axis=1)
    k = keep & (smax <= np.quantile(smax[keep], 0.999))
    if k.any(): keep = k
    dist = np.linalg.norm(means - np.median(means[keep], axis=0), axis=1)
    k = keep & (dist <= np.quantile(dist[keep], 0.999))
    if k.any(): keep = k
    return keep

def write_splat(path, means, log_scales, quats, sh0, logit_op):
    n = len(means)
    order = np.argsort(-np.exp(log_scales.sum(axis=1)) / (1.0 + np.exp(-logit_op)))
    buf = np.zeros(n, dtype=[("pos", "<f4", 3), ("scale", "<f4", 3),
                             ("rgba", "u1", 4), ("rot", "u1", 4)])
    buf["pos"] = means[order]
    buf["scale"] = np.exp(log_scales[order])
    rgb = 0.5 + SH_C0 * sh0[order]
    a = 1.0 / (1.0 + np.exp(-logit_op[order]))
    buf["rgba"] = (np.concatenate([rgb, a[:, None]], axis=1) * 255).clip(0, 255).astype(np.uint8)
    q = quats[order] / np.linalg.norm(quats[order], axis=1, keepdims=True)
    buf["rot"] = (q * 128 + 128).clip(0, 255).astype(np.uint8)
    buf.tofile(path)

def write_ply(path, means, log_scales, quats, sh0, shN, logit_op):
    n = len(means)
    rest = shN.transpose(0, 2, 1).reshape(n, -1) if shN is not None else np.zeros((n, 0), np.float32)
    names = (["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
             + [f"f_rest_{i}" for i in range(rest.shape[1])]
             + ["opacity", "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"])
    cols = [means, np.zeros((n, 3), np.float32), sh0, rest, logit_op[:, None], log_scales, quats]
    data = np.concatenate(cols, axis=1).astype("<f4")
    header = ("ply\nformat binary_little_endian 1.0\n"
              f"element vertex {n}\n"
              + "".join(f"property float {k}\n" for k in names) + "end_header\n")
    with open(path, "wb") as f:
        f.write(header.encode())
        f.write(data.tobytes())

# ───────── Đọc checkpoint cuối ─────────
ckpts = sorted(glob.glob(f"{GSPLAT_OUTPUT}/**/ckpts/*.pt", recursive=True),
               key=lambda p: int(re.search(r"ckpt_(\d+)", p).group(1)))
assert ckpts, "Không có checkpoint - training chưa chạy hoặc bị lỗi"
step = int(re.search(r"ckpt_(\d+)", ckpts[-1]).group(1))
print("Checkpoint:", ckpts[-1], "| step", step)
assert step >= 30000, f"Checkpoint chỉ ở step {step} - có vẻ là lần train cũ!"

sp = torch.load(ckpts[-1], map_location="cpu", weights_only=False)["splats"]
N        = sp["means"].shape[0]
means    = sp["means"].numpy().astype(np.float32)
scales   = sp["scales"].numpy().astype(np.float32)
quats    = sp["quats"].numpy().astype(np.float32)
logit_op = sp["opacities"].numpy().reshape(-1).astype(np.float32)
sh0      = sp["sh0"].numpy().reshape(N, 3).astype(np.float32)
shN      = sp["shN"].numpy().astype(np.float32) if "shN" in sp else None

_op = 1.0 / (1.0 + np.exp(-np.clip(logit_op, -30, 30)))
print(f"Gaussians: {N:,} | opacity min={np.nanmin(_op):.4f} median={np.nanmedian(_op):.4f} max={np.nanmax(_op):.4f}")

keep = prune_mask(means, scales, logit_op)
if keep.sum() == 0:
    raise RuntimeError("Mọi Gaussian đều NaN/Inf -> training bị phân kỳ.")
print(f"Gaussians: {N:,} -> giữ {keep.sum():,} (bỏ {N-keep.sum():,})")

PLY_PATH   = f"{OUT_DIR}/scene.ply"
SPLAT_PATH = f"{VIEWER_DIR}/scene.splat"
write_ply(PLY_PATH, means[keep], scales[keep], quats[keep], sh0[keep],
          shN[keep] if shN is not None else None, logit_op[keep])
write_splat(SPLAT_PATH, means[keep], scales[keep], quats[keep], sh0[keep], logit_op[keep])
print(f"✓ {PLY_PATH}   ({os.path.getsize(PLY_PATH)/1e6:.1f} MB)")
print(f"✓ {SPLAT_PATH} ({os.path.getsize(SPLAT_PATH)/1e6:.1f} MB)")

# ───────── Chất lượng held-out (trên checkpoint chưa lọc) ─────────
stats = sorted(glob.glob(f"{GSPLAT_OUTPUT}/stats/val_step*.json"),
               key=lambda p: int(re.search(r"val_step(\d+)", p).group(1)))
if stats:
    m = json.load(open(stats[-1]))
    print(f"\nHeld-out ({os.path.basename(stats[-1])}): PSNR {m.get('psnr', float('nan')):.2f} dB | "
          f"SSIM {m.get('ssim', float('nan')):.3f} | LPIPS {m.get('lpips', float('nan')):.3f}")

# ───────── Viewer web ─────────
VIEWER_HTML = r'''<!doctype html>
<html lang="vi">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SpatialScene3D - 3D Viewer</title>
<style>
  html, body { margin: 0; height: 100%; background: #111; overflow: hidden;
               font-family: system-ui, sans-serif; color: #eee; }
  #ui { position: fixed; top: 12px; left: 12px; z-index: 10; max-width: 340px;
        background: rgba(0,0,0,.6); padding: 10px 12px; border-radius: 8px; font-size: 13px; }
  #ui button, .btn { margin: 4px 4px 0 0; padding: 4px 8px; border: 0; border-radius: 4px;
               background: #2d6cdf; color: #fff; cursor: pointer; font-size: 13px; display: inline-block; }
  #hint { opacity: .7; margin-top: 8px; line-height: 1.4; }
  #dropzone { position: fixed; inset: 0; z-index: 30; display: none; align-items: center;
              justify-content: center; background: rgba(45,108,223,.25);
              border: 3px dashed #2d6cdf; font-size: 22px; pointer-events: none; }
  #dropzone.on { display: flex; }
</style>
</head>
<body>
<div id="dropzone">Thả file .splat / .ply vào đây</div>
<div id="ui">
  <b>SpatialScene3D</b>
  <div id="status">Đang khởi tạo...</div>
  <div>
    <button data-a="x+">Xoay X +90°</button>
    <button data-a="x-">Xoay X -90°</button>
    <button data-a="y+">Xoay Y +90°</button>
    <button data-a="z+">Xoay Z +90°</button>
    <button data-a="flip">Lật 180°</button>
    <button data-a="reset">Reset</button>
  </div>
  <div>
    <button id="zin">Zoom +</button>
    <button id="zout">Zoom −</button>
    <label class="btn">Chọn file khác<input id="file" type="file" accept=".splat,.ply" hidden></label>
  </div>
  <div id="hint">Chuột trái: xoay · Cuộn: zoom · Chuột phải (hoặc Shift + kéo): dịch.<br>
  Cảm ứng: 1 ngón xoay, 2 ngón zoom/dịch. Phím + / − để zoom.<br>
  Kéo thả file .splat / .ply vào cửa sổ để xem file khác.</div>
</div>
<script type="module">
import * as SPLAT from "https://cdn.jsdelivr.net/npm/gsplat@1.2.9/dist/index.es.js";
const $ = (id) => document.getElementById(id);
const statusEl = $("status");
const renderer = new SPLAT.WebGLRenderer();
const scene    = new SPLAT.Scene();
const camera   = new SPLAT.Camera();
const controls = new SPLAT.OrbitControls(camera, renderer.canvas);
controls.minZoom = 0.01; controls.maxZoom = 500; controls.zoomSpeed = 1.5;
controls.panSpeed = 2.0; controls.orbitSpeed = 1.0; controls.dampening = 0.15;

function zoomBy(dy) {
  renderer.canvas.dispatchEvent(new WheelEvent("wheel", { deltaY: dy, bubbles: true, cancelable: true }));
}
let splat = null;
const ang = { x: 0, y: 0, z: 0 };
function applyRotation() {
  if (!splat) return;
  splat.rotation = SPLAT.Quaternion.FromEuler(new SPLAT.Vector3(ang.x, ang.y, ang.z));
}
function onLoaded(s) {
  splat = s; ang.x = ang.y = ang.z = 0; applyRotation();
  statusEl.textContent = splat.data.vertexCount.toLocaleString() + " Gaussians";
}
const prog = (p) => { statusEl.textContent = "Đang tải " + Math.round(p * 100) + "%"; };
async function loadFile(file) {
  scene.reset();
  statusEl.textContent = "Đang đọc " + file.name + "...";
  try {
    const s = file.name.toLowerCase().endsWith(".ply")
      ? await SPLAT.PLYLoader.LoadFromFileAsync(file, scene, prog)
      : await SPLAT.Loader.LoadFromFileAsync(file, scene, prog);
    onLoaded(s);
  } catch (err) { statusEl.textContent = "Lỗi: " + err.message; }
}
async function autoLoad() {
  if (location.protocol === "file:") {
    statusEl.textContent = "Mở bằng file:// - hãy kéo thả scene.splat vào cửa sổ.";
    return;
  }
  try {
    statusEl.textContent = "Đang tải scene...";
    const s = await SPLAT.Loader.LoadAsync("scene.splat?t=" + Date.now(), scene, prog);
    onLoaded(s);
  } catch (err) {
    statusEl.textContent = "Không tải được scene.splat: " + err.message;
  }
}
let depth = 0;
window.addEventListener("dragenter", (e) => { e.preventDefault(); depth++; $("dropzone").classList.add("on"); });
window.addEventListener("dragleave", (e) => { e.preventDefault(); if (--depth <= 0) { depth = 0; $("dropzone").classList.remove("on"); } });
window.addEventListener("dragover",  (e) => e.preventDefault());
window.addEventListener("drop", (e) => {
  e.preventDefault(); depth = 0; $("dropzone").classList.remove("on");
  const f = e.dataTransfer.files[0]; if (f) loadFile(f);
});
$("file").addEventListener("change", (e) => { const f = e.target.files[0]; if (f) loadFile(f); e.target.value = ""; });
document.querySelectorAll("button[data-a]").forEach((b) => {
  b.addEventListener("click", () => {
    const a = b.dataset.a, q = Math.PI / 2;
    if (a === "x+") ang.x += q; else if (a === "x-") ang.x -= q;
    else if (a === "y+") ang.y += q; else if (a === "z+") ang.z += q;
    else if (a === "flip") ang.x += Math.PI; else { ang.x = 0; ang.y = 0; ang.z = 0; }
    applyRotation();
  });
});
$("zin").addEventListener("click",  () => zoomBy(-300));
$("zout").addEventListener("click", () => zoomBy(300));
window.addEventListener("keydown", (e) => {
  if (e.key === "+" || e.key === "=") zoomBy(-200);
  if (e.key === "-") zoomBy(200);
});
autoLoad();
const frame = () => { controls.update(); renderer.resize(); renderer.render(scene, camera); requestAnimationFrame(frame); };
requestAnimationFrame(frame);
</script>
</body>
</html>
'''
open(f"{VIEWER_DIR}/index.html", "w").write(VIEWER_HTML)
shutil.make_archive(f"{OUT_DIR}/viewer", "zip", VIEWER_DIR)
print(f"✓ Viewer: {VIEWER_DIR}/index.html | Gói tải về: {OUT_DIR}/viewer.zip")

# ───────── Bật server (cổng 8010 để không đụng server cũ ở 8000) ─────────
PORT = 8010
subprocess.run("fuser -k %d/tcp" % PORT, shell=True, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "http.server"], stderr=subprocess.DEVNULL)
time.sleep(1)
subprocess.Popen([sys.executable, "-m", "http.server", str(PORT), "--bind", "0.0.0.0",
                  "--directory", VIEWER_DIR],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)

print("\n" + "=" * 70)
if IN_COLAB:
    from google.colab import output
    print("Viewer hiện bên dưới. Kéo thả file .splat/.ply vào khung để xem file khác.")
    print("=" * 70)
    output.serve_kernel_port_as_iframe(PORT, path="/index.html", height=800)
else:
    print(f"Mở trình duyệt: http://localhost:{PORT}/")
    print("=" * 70)

Checkpoint: /content/scene3d_work/scene1/gsplat_output/ckpts/ckpt_39999_rank0.pt | step 39999
Gaussians: 3,500,000 | opacity min=0.0000 median=0.0120 max=0.9996
Gaussians: 3,500,000 -> giữ 2,016,234 (bỏ 1,483,766)
✓ /content/outputs/scene1/scene.ply   (500.0 MB)
✓ /content/viewer/scene1/scene.splat (64.5 MB)

Held-out (val_step39999.json): PSNR 39.97 dB | SSIM 0.982 | LPIPS 0.037
✓ Viewer: /content/viewer/scene1/index.html | Gói tải về: /content/outputs/scene1/viewer.zip

Viewer hiện bên dưới. Kéo thả file .splat/.ply vào khung để xem file khác.


<IPython.core.display.Javascript object>